In [3]:
!git clone https://github.com/poweredbyfreelancer/Boom-Challenge-Datasets.git

Cloning into 'Boom-Challenge-Datasets'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 39 (delta 10), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 386.74 KiB | 2.80 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [4]:
import os

os.listdir("Boom-Challenge-Datasets")

['inverse_design1',
 '.git',
 'forward_prediction',
 'README.md',
 'inverse_design']

In [5]:
import pandas as pd

train = pd.read_csv("Boom-Challenge-Datasets/forward_prediction/train.csv")
labels = pd.read_csv("Boom-Challenge-Datasets/forward_prediction/train_labels.csv")

test = pd.read_csv("Boom-Challenge-Datasets/forward_prediction/test.csv")

In [6]:
X = train.values
y = labels.values

In [7]:
!git clone https://github.com/poweredbyfreelancer/Boom-Challenge-Datasets.git

fatal: destination path 'Boom-Challenge-Datasets' already exists and is not an empty directory.


In [8]:
import pandas as pd

train = pd.read_csv("Boom-Challenge-Datasets/forward_prediction/train.csv")
labels = pd.read_csv("Boom-Challenge-Datasets/forward_prediction/train_labels.csv")
test = pd.read_csv("Boom-Challenge-Datasets/forward_prediction/test.csv")

X = train.values
y = labels.values

In [9]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X = scaler_X.fit_transform(X)
y = scaler_y.fit_transform(y)

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

In [12]:
import torch.nn as nn

class ImpactModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 6)
        )

    def forward(self, x):
        return self.net(x)

In [13]:
def physics_loss(x, y_pred):
    energy = x[:, 0]

    p80 = y_pred[:, 0]
    r95 = y_pred[:, 3]

    loss = torch.mean(torch.relu(energy - p80)) + torch.mean(torch.relu(energy - r95))
    return loss

In [14]:
model = ImpactModel()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
mse = nn.MSELoss()

lambda_phys = 0.1

In [15]:
for epoch in range(50):
    model.train()

    pred = model(X_train)

    loss_data = mse(pred, y_train)
    loss_phys = physics_loss(X_train, pred)

    loss = loss_data + lambda_phys * loss_phys

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} Loss {loss.item():.4f}")

Epoch 0 Loss 1.1012
Epoch 10 Loss 0.9335
Epoch 20 Loss 0.6532
Epoch 30 Loss 0.4528
Epoch 40 Loss 0.3470


In [16]:
model.eval()
with torch.no_grad():
    val_pred = model(X_val)
    val_loss = mse(val_pred, y_val)

print("Validation Loss:", val_loss.item())

Validation Loss: 0.20028457045555115


In [17]:
with torch.no_grad():
    test_tensor = torch.tensor(scaler_X.transform(test.values), dtype=torch.float32)
    preds = model(test_tensor)

preds = scaler_y.inverse_transform(preds.numpy())

In [18]:
submission = pd.DataFrame(preds, columns=[
    "P80","fines_frac","oversize_frac","R95","R50_fines","R50_oversize"
])

submission.insert(0, "scenario_id", range(len(submission)))

submission.to_csv("prediction_submission.csv", index=False)

In [19]:
import json
import pandas as pd
import numpy as np

with open("Boom-Challenge-Datasets/inverse_design/constraints.json", "r") as f:
    constraints = json.load(f)

constraints

{'constraints': {'p80_min': 96.0, 'p80_max': 101.0, 'r95_max': 175.0},
 'input_bounds': {'energy': {'min': 0.5, 'max': 5.0},
  'angle_rad': {'min': 0.2617993877991494, 'max': 1.5707963267948966},
  'coupling': {'min': 0.2, 'max': 1.7},
  'strength': {'min': 0.4, 'max': 4.2},
  'porosity': {'min': 0.0, 'max': 0.33},
  'gravity': {'min': 1.02, 'max': 10.47},
  'atmosphere': {'min': 0.0, 'max': 1.0},
  'shape_factor': {'min': 0.7, 'max': 1.5}}}

In [20]:
import json

with open("Boom-Challenge-Datasets/inverse_design/constraints.json", "r") as f:
    constraints = json.load(f)

print(constraints)

{'constraints': {'p80_min': 96.0, 'p80_max': 101.0, 'r95_max': 175.0}, 'input_bounds': {'energy': {'min': 0.5, 'max': 5.0}, 'angle_rad': {'min': 0.2617993877991494, 'max': 1.5707963267948966}, 'coupling': {'min': 0.2, 'max': 1.7}, 'strength': {'min': 0.4, 'max': 4.2}, 'porosity': {'min': 0.0, 'max': 0.33}, 'gravity': {'min': 1.02, 'max': 10.47}, 'atmosphere': {'min': 0.0, 'max': 1.0}, 'shape_factor': {'min': 0.7, 'max': 1.5}}}


In [21]:
bounds = constraints["input_bounds"]

def get_min_max(param):
    v = bounds[param]
    return v["min"], v["max"]

In [22]:
def sample_scenarios(n):
    scenarios = []

    for _ in range(n):

        energy_min, energy_max = get_min_max("energy")
        angle_min, angle_max = get_min_max("angle_rad")
        coupling_min, coupling_max = get_min_max("coupling")
        strength_min, strength_max = get_min_max("strength")
        porosity_min, porosity_max = get_min_max("porosity")
        gravity_min, gravity_max = get_min_max("gravity")
        atmosphere_min, atmosphere_max = get_min_max("atmosphere")
        shape_min, shape_max = get_min_max("shape_factor")

        scenario = [
            np.random.uniform(energy_min, energy_max),
            np.random.uniform(angle_min, angle_max),
            np.random.uniform(coupling_min, coupling_max),
            np.random.uniform(strength_min, strength_max),
            np.random.uniform(porosity_min, porosity_max),
            np.random.uniform(gravity_min, gravity_max),
            np.random.uniform(atmosphere_min, atmosphere_max),
            np.random.uniform(shape_min, shape_max),
        ]

        scenarios.append(scenario)

    return np.array(scenarios)

In [23]:
candidates = sample_scenarios(20000)
print(candidates.shape)

(20000, 8)


In [24]:
candidates = sample_scenarios(20000)

print(candidates.shape)

(20000, 8)


In [25]:
candidates_scaled = scaler_X.transform(candidates)
candidates_tensor = torch.tensor(candidates_scaled, dtype=torch.float32)

model.eval()
with torch.no_grad():
    preds = model(candidates_tensor).numpy()

preds = scaler_y.inverse_transform(preds)

In [26]:
mask = (
    (preds[:, 0] >= 96) & (preds[:, 0] <= 101) &
    (preds[:, 3] <= 175)
)

valid_candidates = candidates[mask]
valid_preds = preds[mask]

print("Valid candidates:", len(valid_candidates))

Valid candidates: 32


In [27]:
energy = valid_candidates[:, 0]
p80 = valid_preds[:, 0]
r95 = valid_preds[:, 3]

score = (
    -0.4 * energy
    -0.4 * r95
    -0.2 * np.abs(p80 - 98.5)
)

In [28]:
if len(valid_candidates) >= 20:
    top_idx = np.argsort(score)[-20:]
    chosen_candidates = valid_candidates[top_idx]
else:
    top_idx = np.argsort(score)[-20:]
    chosen_candidates = candidates[top_idx]

print("Selected candidates:", len(chosen_candidates))

Selected candidates: 20


In [29]:
import pandas as pd

submission = pd.DataFrame(chosen_candidates, columns=[
    "energy",
    "angle_rad",
    "coupling",
    "strength",
    "porosity",
    "gravity",
    "atmosphere",
    "shape_factor"
])

submission.insert(0, "submission_id", range(len(submission)))

submission.to_csv("design_submission.csv", index=False)

print("Saved design_submission.csv")

Saved design_submission.csv


In [30]:
from google.colab import files

files.download("design_submission.csv")
files.download("prediction_submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>